# 365 Probabilidades - Dia #108
## Qual a probabilidade de você esquecer o que estudou ontem?

**Tipo:** Aprendizado
**Data de publicação:** 2026-09-29
**Ferramenta:** Python
**Decisão analisada:** Estudar tudo de uma vez ou revisar depois? E depois quando?
**Hashtag:** #365Probabilidades #Dia108

---

### 📖 A História

Eu fiz faculdade de Estatística. Passei em todas as provas. E não lembro de quase nenhuma.

Lembro do que usei depois, todos os dias, durante dezoito anos: distribuição, intervalo, o jeito de desconfiar de um número. Isso ficou porque voltou. Mil vezes.

O resto, aquilo que eu estudava na véspera, numa sentada, com café e a apostila inteira aberta: passou na prova e sumiu na semana seguinte. Como se nunca tivesse entrado.

Depois veio o MBA. Depois o mestrado. A técnica não mudou muito: pilha na véspera, nota boa, memória vazia.

Sempre achei que era falta de memória. Era falta de intervalo.

---

### 📚 O Conceito: o efeito de espaçamento

É um dos achados mais antigos e mais replicados da psicologia: Ebbinghaus já o descrevia em 1885. Duas sessões de estudo separadas por um intervalo produzem mais lembrança do que as mesmas duas sessões coladas. A quantidade de estudo é a mesma. O que muda é o vazio entre elas.

Por um século a pergunta foi "espaçar ajuda?". A resposta virou consenso: ajuda. A pergunta boa é outra: **quanto** espaçar. E aí a resposta depende de uma coisa que quase ninguém pergunta ao aluno: **quando você precisa lembrar?**

Em 2008, um grupo de San Diego respondeu isso com 1.354 pessoas, 32 fatos obscuros e um ano de espera. O intervalo certo entre aprender e revisar não é um número fixo. É uma fração do prazo. Revisar cedo demais desperdiça a revisão. Revisar tarde demais é reaprender do zero.

---

### 🧮 O Modelo

Três camadas, todas com conta própria:

1. **Tradução do efeito** da meta-análise de 2006: de cada 100 coisas estudadas de uma vez, quantas chegam à prova, e quantas chegam quando o estudo é espaçado.
2. **Posterior de Jeffreys** sobre a contagem de experimentos em que espaçar venceu (259 de 271).
3. **A crista**: o intervalo ótimo em função do prazo, interpolado por mim entre os quatro pontos do Cepeda 2008, com a incerteza declarada de o ótimo ter sido medido numa grade de dias, e não num contínuo. E um teste: uma regra única (lei de potência) descreve os quatro pontos? Não. A crista achata.

**Fontes:**
- Cepeda, N. J., Pashler, H., Vul, E., Wixted, J. T. & Rohrer, D. (2006). *Distributed practice in verbal recall tasks: A review and quantitative synthesis.* Psychological Bulletin, 132(3), 354-380. Meta-análise: **839 avaliações em 317 experimentos de 184 artigos.** Espaçado venceu o concentrado em **259 de 271** comparações; **36,7% contra 47,3%** de acerto no teste final (d ≈ 0,69). O intervalo que maximiza a lembrança cresce com o prazo até o teste.
- Cepeda, N. J., Vul, E., Rohrer, D., Wixted, J. T. & Pashler, H. (2008). *Spacing effects in learning: A temporal ridgeline of optimal retention.* Psychological Science, 19(11), 1095-1102. **N = 1.354**, 32 fatos, 26 combinações de intervalo (0 a 105 dias) e prazo (7, 35, 70 e 350 dias); 23 a 79 pessoas por célula. Intervalos ótimos observados: **1, 11, 21 e 21 dias.** Ganho de lembrança do intervalo ótimo sobre o intervalo zero: **10%, 59%, 111% e 77%** (d = 1,1 no agregado).

**Nota metodológica:** ×0,80 não se aplica. Lembrança medida em teste com gabarito, não autorrelato.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

plt.rcParams['font.family'] = 'serif'
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'

DOURADO = '#c9a14a'
VERMELHO = '#c0392b'
VERDE = '#2a8a82'
CINZA = '#6b6a64'

def br(x, dec=0):
    # número em formato brasileiro
    s = f"{x:,.{dec}f}"
    return s.replace(',', 'X').replace('.', ',').replace('X', '.')

print("✅ Bibliotecas carregadas")


In [ ]:
# --- DADOS DA LITERATURA ---

# CEPEDA, PASHLER, VUL, WIXTED & ROHRER 2006 - Psychological Bulletin 132(3)
n_avaliacoes = 839
n_experimentos = 317
n_artigos = 184
venceu_espacado = 259       # comparações em que espaçado > concentrado
comparacoes = 271
p_concentrado = 0.367       # acerto no teste final, estudo concentrado (massed)
p_espacado = 0.473          # acerto no teste final, estudo espaçado
d_meta = 0.69               # tamanho de efeito reportado

# CEPEDA, VUL, ROHRER, WIXTED & PASHLER 2008 - Psychological Science 19(11)
n_2008 = 1354
n_fatos = 32
prazos = np.array([7, 35, 70, 350])              # dias até o teste (RI)
grades = {                                        # intervalos testados em cada prazo
    7:   [0, 1, 2, 7, 21, 105],
    35:  [0, 1, 4, 7, 11, 21, 105],
    70:  [0, 1, 7, 14, 21, 105],
    350: [0, 1, 7, 21, 35, 70, 105],
}
otimo_obs = np.array([1, 11, 21, 21])            # intervalo ótimo observado (lembrança)
ganho_lembranca = np.array([10, 59, 111, 77])    # % de ganho: ótimo vs intervalo zero
ganho_reconhecimento = np.array([1, 10, 31, 60])
d_2008 = 1.1                                     # ótimo vs zero, agregado (lembrança)
n_por_celula = (23, 79)

print("=" * 65)
print("  DADOS DA LITERATURA - O EFEITO DE ESPAÇAMENTO")
print("=" * 65)
print(f"\n  Cepeda et al. 2006 (meta-análise):")
print(f"  → {n_avaliacoes} avaliações, {n_experimentos} experimentos, {n_artigos} artigos")
print(f"  → espaçado venceu em {venceu_espacado} de {comparacoes} comparações")
print(f"  → acerto final: concentrado {p_concentrado*100:.1f}%  |  espaçado {p_espacado*100:.1f}%  (d = {d_meta})")
print(f"\n  Cepeda et al. 2008 (N = {br(n_2008)}, {n_fatos} fatos, {n_por_celula[0]} a {n_por_celula[1]} por célula):")
print(f"  {'prazo':>8} {'ótimo':>8} {'razão':>8} {'ganho':>8}")
for ri, g, ga in zip(prazos, otimo_obs, ganho_lembranca):
    print(f"  {ri:>6} d {g:>6} d {g/ri*100:>7.0f}% {ga:>7}%")
print("=" * 65)


In [ ]:
# --- O MODELO ---
rng = np.random.default_rng(42)

# ── CAMADA 1: tradução do efeito (de cada 100 coisas estudadas) ──
esquece_concentrado = 100 * (1 - p_concentrado)
esquece_espacado = 100 * (1 - p_espacado)
salvas_pelo_intervalo = esquece_concentrado - esquece_espacado
# probabilidade de uma pessoa que espaçou lembrar mais do que uma que concentrou (CLES)
cles = stats.norm.cdf(d_meta / np.sqrt(2))

# ── CAMADA 2: posterior de Jeffreys sobre "espaçar vence" ──
post_vence = stats.beta(venceu_espacado + 0.5, comparacoes - venceu_espacado + 0.5)
p_vence = post_vence.mean()
ic_vence = post_vence.interval(0.95)

# ── CAMADA 3: a crista - intervalo ótimo em função do prazo ──
# Razão observada ótimo/prazo em cada um dos quatro prazos
razao_obs = otimo_obs / prazos

# Incerteza de grade: o ótimo "observado" é só o melhor ponto testado.
# O ótimo verdadeiro está em algum lugar entre os vizinhos da grade, mais perto do
# ponto que venceu do que dos que perderam. Premissa declarada: em escala log,
# distribuição triangular com moda no ótimo observado e apoio nos pontos médios
# geométricos até o vizinho anterior e o seguinte.
def faixa_otimo(ri, g):
    grade = grades[ri]
    i = grade.index(g)
    ant = max(grade[i-1], 0.5) if i > 0 else g      # 0,5 dia como piso: mesmo dia é "intervalo zero"
    seg = grade[i+1] if i < len(grade)-1 else g
    return np.sqrt(ant * g), np.sqrt(g * seg)

faixas = np.array([faixa_otimo(ri, g) for ri, g in zip(prazos, otimo_obs)])

def sorteia_otimos(rng):
    return np.exp(rng.triangular(np.log(faixas[:, 0]), np.log(otimo_obs), np.log(faixas[:, 1])))

# A crista: interpolação em log-log entre os quatro prazos, sem forma imposta,
# com o ótimo forçado a não diminuir quando o prazo cresce (monótona).
# Além de 350 dias, mantém o último valor (declarado).
prazos_uso = np.array([3, 7, 14, 30, 60, 90, 180, 365])
def crista(ot, pz):
    ot = np.maximum.accumulate(ot)                     # monótona
    lp = np.log(pz); lx = np.log(prazos); ly = np.log(ot)
    y = np.interp(lp, lx, ly, left=np.nan, right=ly[-1])
    # antes de 7 dias: mesma razão do ponto de 7 dias
    y = np.where(np.isnan(y), np.log(ot[0] / prazos[0] * pz), y)
    return np.exp(y)

N_SIM = 20_000
otimo_uso = np.empty((N_SIM, len(prazos_uso)))
for k in range(N_SIM):
    otimo_uso[k] = crista(sorteia_otimos(rng), prazos_uso)
otimo_med = np.median(otimo_uso, axis=0)
otimo_lo, otimo_hi = np.percentile(otimo_uso, [2.5, 97.5], axis=0)
razao_med = otimo_med / prazos_uso
p_otimo_maior_que_1 = (otimo_uso > 1).mean(axis=0)   # "estudei ontem, revisar hoje é cedo?"

# Teste de uma regra única: uma lei de potência (reta em log-log) descreve os 4 pontos?
b_hat, ln_a = np.polyfit(np.log(prazos), np.log(otimo_obs), 1); a_hat = np.exp(ln_a)
pred = a_hat * prazos ** b_hat
r2 = 1 - np.sum((np.log(otimo_obs) - np.log(pred))**2) / np.sum((np.log(otimo_obs) - np.log(otimo_obs).mean())**2)

print("=" * 65)
print("  MODELO - O QUE VOCÊ ESQUECE E QUANDO REVISAR")
print("=" * 65)
print(f"\n  CAMADA 1 - de cada 100 coisas estudadas:")
print(f"  → numa sentada, chegam à prova {100-esquece_concentrado:.0f}; somem {esquece_concentrado:.0f}")
print(f"  → espaçadas, chegam {100-esquece_espacado:.0f}; somem {esquece_espacado:.0f}")
print(f"  → o intervalo salva {salvas_pelo_intervalo:.0f} de cada 100")
print(f"  → P(quem espaçou lembra mais do que quem concentrou) = {cles*100:.0f}%")
print(f"\n  CAMADA 2 - espaçar vence ({venceu_espacado} de {comparacoes}):")
print(f"  → posterior de Jeffreys: {p_vence*100:.1f}%  IC 95% [{ic_vence[0]*100:.1f}%, {ic_vence[1]*100:.1f}%]")
print(f"\n  CAMADA 3 - a crista (razão ótimo/prazo observada):")
for ri, g, rz in zip(prazos, otimo_obs, razao_obs):
    print(f"  → prova em {ri:>3} dias: revisar no dia {g:>2}  ({rz*100:.0f}% do prazo)")
print(f"\n  Tabela de uso (interpolação com incerteza de grade, {br(N_SIM)} sorteios):")
print(f"  {'prova em':>10} {'revise em':>12} {'faixa 95%':>16} {'razão':>7} {'P(ótimo>1d)':>12}")
for pz, m, lo, hi, rz, p1 in zip(prazos_uso, otimo_med, otimo_lo, otimo_hi, razao_med, p_otimo_maior_que_1):
    print(f"  {pz:>7} d  {m:>8.1f} d   [{lo:>5.1f}, {hi:>5.1f}] d {rz*100:>6.0f}% {p1*100:>10.0f}%")
print(f"\n  Uma regra única serve? lei de potência ótimo = {a_hat:.2f} · prazo^{b_hat:.2f}")
print(f"  → R² em log = {r2:.2f}; prevê {pred[2]:.0f} dias para a prova de 70 (observado 21)")
print(f"     e {pred[3]:.0f} dias para a de 350 (observado 21). A crista achata depois de 70 dias.")
print("=" * 65)


In [ ]:
# --- VISUALIZAÇÃO - GRÁFICOS SEPARADOS ---

# ── GRÁFICO 1 - de cada 100 coisas estudadas ──
fig1, ax1 = plt.subplots(figsize=(10, 6))
cats = ['Estudo concentrado\n(tudo de uma vez)', 'Estudo espaçado\n(com intervalo)']
lembra = [100 - esquece_concentrado, 100 - esquece_espacado]
esquece = [esquece_concentrado, esquece_espacado]
x = np.arange(2)
ax1.bar(x, lembra, width=0.5, color=VERDE, alpha=0.9, label='chegam à prova')
ax1.bar(x, esquece, width=0.5, bottom=lembra, color=VERMELHO, alpha=0.75, label='somem')
for i in range(2):
    ax1.text(x[i], lembra[i]/2, f'{lembra[i]:.0f}', ha='center', va='center', color='white', fontsize=22, fontweight='bold')
    ax1.text(x[i], lembra[i] + esquece[i]/2, f'{esquece[i]:.0f}', ha='center', va='center', color='white', fontsize=22, fontweight='bold')
ax1.set_xticks(x); ax1.set_xticklabels(cats, fontsize=12)
ax1.set_ylim(0, 108); ax1.set_ylabel('de cada 100 coisas estudadas')
ax1.legend(loc='upper right', frameon=False)
ax1.set_title(f'O intervalo salva {salvas_pelo_intervalo:.0f} de cada 100\n839 avaliações, 317 experimentos (Cepeda et al. 2006)', fontsize=13, pad=14)
plt.figtext(0.5, 0.01, 'Fonte: Cepeda, Pashler, Vul, Wixted & Rohrer 2006, Psychological Bulletin | #365Probabilidades',
            ha='center', fontsize=9, color='gray')
plt.tight_layout()
plt.savefig('dia-108-grafico-01-cem-coisas.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Gráfico 1 salvo!")

# ── GRÁFICO 2 - a crista: quando revisar, em função do prazo (assinatura) ──
fig2, ax2 = plt.subplots(figsize=(10, 6.5))
pz_grid = np.logspace(np.log10(3), np.log10(400), 200)
curvas = np.empty((4000, len(pz_grid)))
for k in range(4000):
    curvas[k] = crista(sorteia_otimos(rng), pz_grid)
lo_c, med_c, hi_c = np.percentile(curvas, [2.5, 50, 97.5], axis=0)
ax2.fill_between(pz_grid, lo_c, hi_c, color=DOURADO, alpha=0.18, label='faixa de 95% (incerteza de grade)')
ax2.plot(pz_grid, med_c, color=DOURADO, lw=3, label='a crista: intervalo ótimo (mediana)')
ax2.plot(pz_grid, 0.20 * pz_grid, ls=':', color=CINZA, lw=1.5, label='regra de 20% do prazo')
ax2.plot(pz_grid, 0.05 * pz_grid, ls='--', color=CINZA, lw=1.5, label='regra de 5% do prazo')
ax2.plot(pz_grid, a_hat * pz_grid ** b_hat, color=VERMELHO, lw=1.2, alpha=0.6,
         label=f'regra única (lei de potência, R² {r2:.2f}): não serve')
ax2.errorbar(prazos, otimo_obs, yerr=[otimo_obs - faixas[:, 0], faixas[:, 1] - otimo_obs],
             fmt='o', color=VERDE, ms=9, capsize=4, lw=1.5, label='ótimo observado (barra: apoio da triangular)', zorder=5)
ax2.set_xscale('log'); ax2.set_yscale('log'); ax2.set_ylim(0.3, 90)
ax2.set_xticks([7, 14, 30, 70, 180, 350]); ax2.set_xticklabels(['7', '14', '30', '70', '180', '350'])
ax2.set_yticks([1, 3, 7, 21, 60]); ax2.set_yticklabels(['1', '3', '7', '21', '60'])
ax2.set_xlabel('prazo até a prova (dias)'); ax2.set_ylabel('intervalo ótimo até a revisão (dias)')
ax2.set_title('A crista: quando revisar depende de quando você precisa lembrar\nCepeda et al. 2008, N = 1.354; interpolação própria com incerteza de grade',
              fontsize=13, pad=14)
ax2.legend(loc='upper left', frameon=False, fontsize=9.5)
plt.figtext(0.5, 0.01, 'Fonte: Cepeda, Vul, Rohrer, Wixted & Pashler 2008, Psychological Science | #365Probabilidades',
            ha='center', fontsize=9, color='gray')
plt.tight_layout()
plt.savefig('dia-108-grafico-02-crista.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Gráfico 2 salvo!")

# ── GRÁFICO 3 - o ganho de revisar na hora certa, por prazo ──
fig3, ax3 = plt.subplots(figsize=(10, 6))
xx = np.arange(4); w = 0.36
b1 = ax3.bar(xx - w/2, ganho_lembranca, w, color=VERDE, alpha=0.9, label='lembrar (recall)')
b2 = ax3.bar(xx + w/2, ganho_reconhecimento, w, color=DOURADO, alpha=0.9, label='reconhecer (múltipla escolha)')
for b, v in zip(list(b1) + list(b2), list(ganho_lembranca) + list(ganho_reconhecimento)):
    ax3.text(b.get_x() + b.get_width()/2, v + 2, f'+{v}%', ha='center', fontsize=12, fontweight='bold')
ax3.set_xticks(xx); ax3.set_xticklabels([f'prova em {p} dias\nrevisão no dia {g}' for p, g in zip(prazos, otimo_obs)], fontsize=11)
ax3.set_ylim(0, 130); ax3.set_ylabel('ganho sobre revisar no mesmo dia (%)')
ax3.axhline(100, color=VERMELHO, ls='--', lw=1.2); ax3.text(3.45, 103, 'dobra', color=VERMELHO, fontsize=10, ha='right')
ax3.legend(loc='upper left', frameon=False)
ax3.set_title('Revisar na hora certa, contra revisar no mesmo dia\nquanto mais longe a prova, maior o ganho (Cepeda et al. 2008)', fontsize=13, pad=14)
ax3.text(0.99, 0.80, 'dias ótimos da lembrança; para reconhecer, foram 1, 7, 7 e 21', transform=ax3.transAxes, ha='right', fontsize=9, color=CINZA)
plt.figtext(0.5, 0.01, 'Fonte: Cepeda, Vul, Rohrer, Wixted & Pashler 2008, Psychological Science | #365Probabilidades',
            ha='center', fontsize=9, color='gray')
plt.tight_layout()
plt.savefig('dia-108-grafico-03-ganho.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Gráfico 3 salvo!")


### 💡 O Insight

**De cada 100 coisas que você estudou ontem, numa sentada, 63 não chegam à prova.**

Com intervalo entre o estudo e a revisão, somem 53. O intervalo sozinho salva 11 de cada 100 (63,3 contra 52,7), sem uma hora a mais de estudo. Em 259 de 271 experimentos, espaçar venceu.

E o intervalo tem tamanho certo. Não é "amanhã". É uma fração do prazo: para uma prova em uma semana, revise amanhã; em um mês, dali a uns nove dias; em dois ou três meses, dali a três semanas. Depois disso a crista achata: mesmo para um ano, o melhor intervalo testado foram as mesmas três semanas. Na hora certa, a lembrança de dois meses depois mais que dobra em relação a revisar no mesmo dia.

Eu revisava tudo na véspera. Ou seja: no intervalo zero, com o prazo mais curto possível. A nota vinha. A memória, não. O que ficou da faculdade foi o que eu usei durante dezoito anos, com intervalos que a vida escolheu por mim.

A pergunta que fica:

*O que você está estudando hoje que precisa lembrar daqui a um ano? E quando você marcou a revisão?*

---

### ⚠️ Limitações do Modelo

- Os 36,7% e 47,3% são médias de 839 avaliações com prazos, materiais e pessoas muito diferentes; o "ontem" da pergunta é uma licença: o teste final varia de segundos a meses. O número responde "até a prova", não "até amanhã".
- A crista é interpolada por mim entre **quatro pontos**, sem forma imposta, e forçada a não diminuir com o prazo. A faixa de 95% vem da premissa de que o ótimo verdadeiro está entre os vizinhos testados na grade, mais perto do que venceu (triangular em escala log), e é minha, não dos autores. Entre 21 e 105 dias, no prazo de 70, nada foi testado. Além de 350 dias não há dado; o modelo repete o último valor.
- "Achata depois de 70 dias" se apoia em dois pontos (70 e 350) com 23 a 79 pessoas por célula; a grade do prazo de um ano testou 21 e 35 dias, e 35 perdeu por pouco. É a leitura mais provável, não uma certeza.
- Os ganhos de 10%, 59%, 111% e 77% são relativos ao intervalo zero; não tive acesso às proporções por célula (só à figura), então não recomputei o intervalo de confiança de cada ganho. Fica como pendência.
- Material: fatos de trivia e listas verbais. Nada aqui é sobre habilidade motora, raciocínio ou o que se aprende fazendo.
- Cepeda 2008 é amostra online, 72% mulheres, idade média 34; 23 a 79 pessoas por célula.

*A ciência é honesta sobre o que não sabe. O modelo também.*

---

### 📎 Links
- Substack: [link do post]
- Instagram: [link do post]

---
*365 Probabilidades · Decidindo com dados, um dia de cada vez.*
